# Feature Engineering
This notebook handles the feature engineering pipiline for this project. It creates a `neighborhood` column utilizing some of the existing location features. It also utilizes the `review_count` feature to create `popularity` tiers and a weighted `ranking_score` for ranking results for the Symbolic baseline mode. Then, it parses the unstructured columns `attributes`, `categories`, and `hours` to create structured fields that can be utilized for the symbolic and neural baseline models.

In [1]:
import re
import ast
import datetime
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
base_path = '/content/drive/My Drive/DS 266/Final Project/data/'
phl_restaurants = pd.read_csv(base_path + 'phl_restaurants_raw.csv')
phl_restaurants.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,rating,review_count,attributes,categories,hours
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."
2,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"{'NoiseLevel': ""u'quiet'"", 'GoodForMeal': ""{'d...","Korean, Restaurants","{'Monday': '11:30-20:30', 'Tuesday': '11:30-20..."
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,901 N Delaware Ave,Philadelphia,PA,19123,39.962582,-75.135657,3.5,65,"{'OutdoorSeating': 'True', 'RestaurantsPriceRa...","Eatertainment, Arts & Entertainment, Brewpubs,...","{'Monday': '0:0-0:0', 'Wednesday': '16:0-22:0'..."
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,3604 Chestnut St,Philadelphia,PA,19104,39.954573,-75.194894,3.0,56,"{'Alcohol': ""u'none'"", 'RestaurantsGoodForGrou...","Restaurants, Automotive, Delis, Gas Stations, ...","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W..."


## Create Neighborhoods Column
The Philadelphia neighborhoods dataset from [OpenDataPhilly](https://opendataphilly.org/datasets/philadelphia-neighborhoods/) will be used to map latitude/longitude coordinates of restaurants in the dataset to their neighborhood.

In [3]:
import geopandas as gpd
from shapely.geometry import Point

# Load Philadelphia neighborhoods GeoJSON file from OpenDataPhilly
neighborhoods = gpd.read_file(base_path + 'philadelphia-neighborhoods.geojson')

# Convert restaurants to GeoDataFrame
gdf = gpd.GeoDataFrame(
    phl_restaurants,
    geometry=gpd.points_from_xy(phl_restaurants.longitude, phl_restaurants.latitude),
    crs='EPSG:4326'
)

# Spatial join to assign each restaurant to the neighborhood polygon it falls within
gdf = gpd.sjoin(gdf, neighborhoods[['geometry', 'MAPNAME']], how='left', predicate='within')

# Insert neighborhoods after longitude column
phl_restaurants['neighborhood'] = gdf['MAPNAME'].values
idx = phl_restaurants.columns.get_loc('longitude') + 1
col = phl_restaurants.pop('neighborhood')
phl_restaurants.insert(idx, 'neighborhood', col)

## Create Popularity Tiers
Bin quintiles of the `review_count` column to create popularity tiers.

In [4]:
display(phl_restaurants['review_count'].quantile([.2, .4, .6, .8, 1.0]))

# Create popularity column by binning review_count into quintiles
phl_restaurants['popularity'] = pd.qcut(
    phl_restaurants['review_count'],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype('Int64')

# Insert popularity after review_count column
idx = phl_restaurants.columns.get_loc('review_count') + 1
col = phl_restaurants.pop('popularity')
phl_restaurants.insert(idx, 'popularity', col)

,review_count
0.2,22.0
0.4,47.0
0.6,100.0
0.8,232.4
1.0,3401.0


## Create Ranking Score
A new column will be engineered that takes a Bayesian weighting of the restaurant's popularity (based on `review_count`) and the its rounded average `rating`. This column can then be used to rank the results from the Symbolic Baseline Model after filtering based on query constraints. The new column will be called `ranking_score`.

In [5]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

C = phl_restaurants['review_count'].median()

rating_norm = scaler.fit_transform(phl_restaurants[['rating']])[:, 0]
weight = phl_restaurants['review_count'] / (phl_restaurants['review_count'] + C)

phl_restaurants['ranking_score'] = (
    weight * rating_norm + (1 - weight) * 0.5
).round(4)

rc_idx = phl_restaurants.columns.get_loc('popularity') + 1
col = phl_restaurants.pop('ranking_score')
phl_restaurants.insert(rc_idx, 'ranking_score', col)

Check updates to the dataset before moving on to engineer the unstructured features.

In [6]:
phl_restaurants.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,neighborhood,rating,review_count,popularity,ranking_score,attributes,categories,hours
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,Chinatown,4.0,80,3,0.6361,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,Old City,4.0,245,5,0.6963,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."
2,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,Hawthorne,4.5,205,4,0.7826,"{'NoiseLevel': ""u'quiet'"", 'GoodForMeal': ""{'d...","Korean, Restaurants","{'Monday': '11:30-20:30', 'Tuesday': '11:30-20..."
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,901 N Delaware Ave,Philadelphia,PA,19123,39.962582,-75.135657,Northern Liberties,3.5,65,3,0.5616,"{'OutdoorSeating': 'True', 'RestaurantsPriceRa...","Eatertainment, Arts & Entertainment, Brewpubs,...","{'Monday': '0:0-0:0', 'Wednesday': '16:0-22:0'..."
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,3604 Chestnut St,Philadelphia,PA,19104,39.954573,-75.194894,University City,3.0,56,3,0.5000,"{'Alcohol': ""u'none'"", 'RestaurantsGoodForGrou...","Restaurants, Automotive, Delis, Gas Stations, ...","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W..."


# Parse Attributes

## Step 1: Parse and flatten attributes

In [7]:
def parse_attr_string(attr_string):
    """Parse an attribute string into a dict, returning {} on failure."""
    if not isinstance(attr_string, str) or pd.isna(attr_string):
        return {}
    try:
        d = ast.literal_eval(attr_string)
        return d if isinstance(d, dict) else {}
    except (ValueError, SyntaxError):
        return {}

In [8]:
def flatten_attrs(attr_string):
    """Parse and flatten nested dict values into parent_subkey format."""
    result = {}
    for key, value in parse_attr_string(attr_string).items():
        if isinstance(value, str) and value.strip().startswith('{'):
            try:
                nested = ast.literal_eval(value)
                if isinstance(nested, dict):
                    for sub_key, sub_val in nested.items():
                        result[f"{key}_{sub_key}"] = sub_val
                    continue
            except (ValueError, SyntaxError):
                pass
        result[key] = value
    return result

## Step 2: Select most common attributes
For the sake of relevance and limiting the number of columns that will be filtered on, only attributes present in at least 25% of rows will be included.

In [9]:
# Count how many rows each top-level attribute key appears in
n_rows = len(phl_restaurants)
min_count = 0.25 * n_rows

top_level_counts = pd.Series(
    [key for attr in phl_restaurants['attributes'] for key in parse_attr_string(attr)]
).value_counts()

selected_attrs = set(top_level_counts[top_level_counts >= min_count].index)
print(f"Threshold: {min_count:.0f} rows (25% of {n_rows})")
print(f"Selected {len(selected_attrs)} attributes: {sorted(selected_attrs)}")

Threshold: 704 rows (25% of 2814)
Selected 22 attributes: ['Alcohol', 'Ambience', 'BikeParking', 'BusinessAcceptsCreditCards', 'BusinessParking', 'Caters', 'DogsAllowed', 'GoodForKids', 'GoodForMeal', 'HappyHour', 'HasTV', 'NoiseLevel', 'OutdoorSeating', 'RestaurantsAttire', 'RestaurantsDelivery', 'RestaurantsGoodForGroups', 'RestaurantsPriceRange2', 'RestaurantsReservations', 'RestaurantsTableService', 'RestaurantsTakeOut', 'WheelchairAccessible', 'WiFi']


In [10]:
# Identify which selected attributes are nested dicts and collect their sub-keys
nested_attrs = {}
for attr_string in phl_restaurants['attributes']:
    for key, value in parse_attr_string(attr_string).items():
        if key in selected_attrs and isinstance(value, str) and value.strip().startswith('{'):
            try:
                nested = ast.literal_eval(value)
                if isinstance(nested, dict):
                    if key not in nested_attrs:
                        nested_attrs[key] = set()
                    nested_attrs[key].update(nested.keys())
            except (ValueError, SyntaxError):
                pass

nested_attr_keys = set(nested_attrs.keys())
print("Nested attributes and their sub-keys:")
for attr, sub_keys in sorted(nested_attrs.items()):
    print(f"  {attr}: {sorted(sub_keys)}")

Nested attributes and their sub-keys:
  Ambience: ['casual', 'classy', 'divey', 'hipster', 'intimate', 'romantic', 'touristy', 'trendy', 'upscale']
  BusinessParking: ['garage', 'lot', 'street', 'valet', 'validated']
  GoodForMeal: ['breakfast', 'brunch', 'dessert', 'dinner', 'latenight', 'lunch']


## Step 3: Encode attributes as binary or ordinal values

In [11]:
def normalize(value):
    """Lowercase + strip quotes/u-prefix from a string value."""
    return str(value).lower().strip("'u'")

In [12]:
def encode_value(key, value):
    """
    Encode a single attribute value:
      - None / missing / unmapped       -> None (becomes NaN)
      - Alcohol: none=0, beer_and_wine/full_bar=1
      - WiFi: no=0, free/paid=1
      - RestaurantsAttire: casual=1, dressy=2, formal=3
      - RestaurantsPriceRange2: 1-4
      - NoiseLevel: quiet=1, average=2, loud=3, very_loud=4
      - True/False booleans or strings  -> 1/0
    """
    if value is None:
        return None

    v = normalize(value)

    if v in ('none', 'nan'):
        return None

    if key == 'Alcohol':
        return {'none': 0, 'beer_and_wine': 1, 'full_bar': 1}.get(v)

    if key == 'WiFi':
        return {'no': 0, 'free': 1, 'paid': 1}.get(v)

    if key == 'RestaurantsAttire':
        return {'casual': 1, 'dressy': 2, 'formal': 3}.get(v)

    if key == 'RestaurantsPriceRange2':
        return int(v) if v in ('1', '2', '3', '4') else None

    if key == 'NoiseLevel':
        return {'quiet': 1, 'average': 2, 'loud': 3, 'very_loud': 4}.get(v)

    if isinstance(value, bool):
        return int(value)
    if v == 'true':
        return 1
    if v == 'false':
        return 0
    try:
        return int(v)
    except ValueError:
        return None

## Step 4: Rename columns

In [13]:
def camel_to_snake(name):
    # exception for WiFi
    if name == 'WiFi':
        return 'has_wifi'
    # rename remaining using snake case
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name).lower()

In [14]:
def clean_col_name(name):
    name = camel_to_snake(name)
    name = name.replace('restaurants_', '').replace('business_', '')
    if name == 'price_range2':
        name = 'price_range'
    if name.startswith('good_for_meal_'):
        name = name.replace('good_for_meal_', 'meal_')
    return name

## Step 5: Build attributes_df

In [15]:
records = []

records = []
for attr_string in phl_restaurants['attributes']:
    flat = flatten_attrs(attr_string)
    row = {}
    for flat_key, value in flat.items():
        base_key = next(
            (p for p in nested_attr_keys if flat_key.startswith(f"{p}_")),
            flat_key
        )
        if base_key in selected_attrs and flat_key not in nested_attr_keys:
            row[flat_key] = encode_value(base_key, value)
    records.append(row)

attributes_df = pd.DataFrame(records)
attributes_df.columns = [clean_col_name(c) for c in attributes_df.columns]
attributes_df = attributes_df.astype('Int64')

attributes_df = pd.concat(
    [phl_restaurants[['business_id', 'name']], attributes_df], axis=1
)

print(attributes_df.shape)
attributes_df.info()

(2814, 41)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2814 entries, 0 to 2813
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   business_id            2814 non-null   object
 1   name                   2814 non-null   object
 2   delivery               2568 non-null   Int64 
 3   outdoor_seating        2291 non-null   Int64 
 4   accepts_credit_cards   2616 non-null   Int64 
 5   parking_garage         2358 non-null   Int64 
 6   parking_street         2417 non-null   Int64 
 7   parking_validated      2410 non-null   Int64 
 8   parking_lot            2483 non-null   Int64 
 9   parking_valet          2548 non-null   Int64 
 10  bike_parking           2229 non-null   Int64 
 11  price_range            2388 non-null   Int64 
 12  take_out               2657 non-null   Int64 
 13  has_wifi               2273 non-null   Int64 
 14  alcohol                833 non-null    Int64 
 15  caters    

In [16]:
attributes_df.head()

,business_id,name,delivery,outdoor_seating,accepts_credit_cards,parking_garage,parking_street,parking_validated,parking_lot,parking_valet,...,meal_latenight,meal_lunch,meal_dinner,meal_brunch,meal_breakfast,noise_level,dogs_allowed,happy_hour,wheelchair_accessible,table_service
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,0,0,0,0,1,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,1,1,1,<NA>,1,<NA>,0,0,...,<NA>,<NA>,1,<NA>,<NA>,2,0,1,1,1
2,ROeacJQwBeh05Rqg7F6TCg,BAP,<NA>,<NA>,1,0,1,0,0,0,...,0,1,1,0,0,1,<NA>,<NA>,<NA>,1
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,1,1,1,0,0,0,1,0,...,0,0,0,0,0,<NA>,1,1,<NA>,<NA>
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,1,0,1,0,1,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2,<NA>,<NA>,<NA>,<NA>


## Step 6: Null percentages per column
For exploratory data analysis, check the percentage of null values in each engineered column. This will be helpful for knowing which columns to use for filtering the for Symbolic Baseline model.

In [17]:
null_pct = attributes_df.select_dtypes(include='Int64').isna().mean() * 100
null_pct.sort_values()

,0
take_out,5.579247
accepts_credit_cards,7.036247
delivery,8.742004
parking_valet,9.452736
parking_lot,11.762615
parking_street,14.108031
parking_validated,14.356787
price_range,15.138593
has_tv,15.245203
parking_garage,16.204691


## Step 7: Merge with existing Restaurants Table

In [18]:
phl_restaurants = phl_restaurants.merge(
    attributes_df, on=['business_id', 'name'], how='left'
)
phl_restaurants.shape

(2814, 55)



---



# Parse Categories

In [19]:
categories_map = pd.read_csv(base_path + 'categories.csv')

## Step 1: Build lookup structures from categories.csv

In [20]:
# Tags to ignore
noise_tags = set(categories_map['noise'].dropna())

# Map each tag -> its column group (dietary_restrictions, cuisine, key_menu, establishment_type)
feature_cols = ['dietary_restrictions', 'cuisine', 'key_menu', 'establishment_type']
tag_to_group = {
    tag: col
    for col in feature_cols
    for tag in categories_map[col].dropna()
}

# Map each cuisine -> its cuisine_group
cuisine_to_group = dict(
    zip(categories_map['cuisine'].dropna(), categories_map['cuisine_group'].dropna())
)

print(f"Noise tags: {len(noise_tags)}")
print(f"Mapped tags: {len(tag_to_group)}")
print(f"Cuisine groups: {sorted(set(cuisine_to_group.values()))}")

Noise tags: 161
Mapped tags: 212
Cuisine groups: ['African', 'American / Fusion', 'East Asian', 'European', 'Latin American / Caribbean', 'Mediterranean / Middle Eastern', 'Slavic', 'South Asian', 'Southeast Asian']


## Step 2: Parse categories column into new feature columns

In [21]:
new_cols = ['dietary_restrictions', 'cuisine_group', 'cuisine', 'key_menu', 'establishment_type']

def parse_categories(cat_string):
    row = {col: [] for col in new_cols}
    if not isinstance(cat_string, str):
        return {col: None for col in new_cols}
    for tag in [t.strip() for t in cat_string.split(',')]:
        if tag in noise_tags or tag not in tag_to_group:
            continue
        group = tag_to_group[tag]
        row[group].append(tag)
        if group == 'cuisine' and tag in cuisine_to_group:
            row['cuisine_group'].append(cuisine_to_group[tag])
    return {
        col: (', '.join(tags) if tags else None)
        for col, tags in row.items()
    }

categories_df = pd.DataFrame(phl_restaurants['categories'].apply(parse_categories).tolist())
categories_df = pd.concat(
    [phl_restaurants[['business_id', 'name']], categories_df], axis=1
)

print(categories_df.shape)
categories_df.info()

(2814, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2814 entries, 0 to 2813
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   business_id           2814 non-null   object
 1   name                  2814 non-null   object
 2   dietary_restrictions  207 non-null    object
 3   cuisine_group         1774 non-null   object
 4   cuisine               1774 non-null   object
 5   key_menu              1884 non-null   object
 6   establishment_type    1346 non-null   object
dtypes: object(7)
memory usage: 154.0+ KB


In [22]:
categories_df.head()

,business_id,name,dietary_restrictions,cuisine_group,cuisine,key_menu,establishment_type
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,None,None,None,"Bubble Tea, Coffee & Tea",Bakeries
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,None,East Asian,Japanese,Sushi Bars,None
2,ROeacJQwBeh05Rqg7F6TCg,BAP,None,East Asian,Korean,None,None
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,None,American / Fusion,American (Traditional),None,"Brewpubs, Bakeries, Breweries"
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,None,None,None,"Coffee & Tea, Sandwiches","Delis, Convenience Stores"


## Step 3: Spot-check coverage and drop any noisy rows remaining

In [23]:
# Rows where every category tag was noise or unmapped (all 5 columns are empty strings)
all_empty = categories_df[new_cols].isna().all(axis=1)
empty_rows = phl_restaurants[all_empty][['business_id', 'name', 'categories']]

print(f"{len(empty_rows)} rows with no mapped category tags:")
display(empty_rows)

18 rows with no mapped category tags:


,business_id,name,categories
125,lqBurWf5LxIhUQ0HtXTu_A,Philadelphia Cricket Club,"Venues & Event Spaces, Restaurants, Active Lif..."
133,mFE9V6LPpsDRUQLEBsBRRA,Pearl of East,Restaurants
249,VD4Rdyj0tauBckrTYgIcpQ,The Concourse at Comcast Center,"Restaurants, Food, Shopping Centers, Shopping"
497,cdZf-iYQRy5D18gTM95aew,The Bellevue Hotel,"Hotels & Travel, Event Planning & Services, Re..."
717,Rzx2E5XgTeGU7FEbUZ-bGg,IKEA,"Kitchen & Bath, Shopping, Restaurants, Furnitu..."
790,x7gEX6dthpJvoOafV1XHaA,Cook,"Adult Education, Specialty Schools, Kitchen Su..."
950,ebuycCvuqc3kn4wSBBOxVQ,The Piazza,"Shopping, Arts & Entertainment, Real Estate, H..."
987,krCwF4raTYvBcCEwcbA88Q,Philadelphia Mills,"Shopping Centers, Shopping, Restaurants, Fashi..."
1040,VJpuuNZ8QXR2zTHIhB0yhw,The Westin Philadelphia,"Hotels & Travel, Venues & Event Spaces, Restau..."
1075,XPiujqYlO9ldyt_hjDjYPw,Pyramid Club,"Nightlife, Caterers, Event Planning & Services..."


In [24]:
ids_to_drop = empty_rows[
    empty_rows['categories'].str.split(',').str.len() > 1
]['business_id'].tolist()

mask = ~phl_restaurants['business_id'].isin(ids_to_drop)
phl_restaurants = phl_restaurants[mask].reset_index(drop=True)
categories_df = categories_df[mask].reset_index(drop=True)

print(f"Dropped {len(ids_to_drop)} rows. Remaining: {len(phl_restaurants)}")

Dropped 15 rows. Remaining: 2799


In [25]:
phl_restaurants.shape

(2799, 55)

In [26]:
# Fraction of rows with at least one tag per column
coverage = categories_df[new_cols].notna().mean() * 100
print("% of rows with at least one tag per column:")
print(coverage.round(1).to_string())

% of rows with at least one tag per column:
dietary_restrictions     7.4
cuisine_group           63.4
cuisine                 63.4
key_menu                67.3
establishment_type      48.1


## Step 4: Merge with existing Restaurants Table

In [27]:
phl_restaurants = phl_restaurants.merge(
    categories_df, on=['business_id', 'name'], how='left'
)
phl_restaurants.shape

(2799, 60)



---



# Parse Hours

## Step 1: Helpers for Days of the Week and 24 Hour Times

In [28]:
DAY_MAP = {
    'Monday': 'mon', 'Tuesday': 'tue', 'Wednesday': 'wed',
    'Thursday': 'thu', 'Friday': 'fri', 'Saturday': 'sat', 'Sunday': 'sun'
}
DAYS_ORDER = ['mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun']
HOURS_COLS = [f"{day}_{oc}" for day in DAYS_ORDER for oc in ('open', 'close')]

def to_decimal(time_str):
    """Convert 'H:MM' string to decimal hours."""
    h, m = map(int, time_str.split(':'))
    return round(h + m / 60, 4)

## Step 2: Parse hours into 14 decimal columns with opening and closing times

In [29]:
def parse_hours(hours_string):
    """Parse an hours dict string into 14 decimal open/close columns.
    Closing times earlier than opening times have 24 added (over-midnight).
    Missing days are filled with None.
    """
    row = {col: None for col in HOURS_COLS}
    if not isinstance(hours_string, str) or pd.isna(hours_string):
        return row
    try:
        hours_dict = ast.literal_eval(hours_string)
    except (ValueError, SyntaxError):
        return row
    for day_full, time_range in hours_dict.items():
        day = DAY_MAP.get(day_full)
        if not day:
            continue
        try:
            open_str, close_str = time_range.split('-')
            open_dec = to_decimal(open_str)
            close_dec = to_decimal(close_str)
            # Handle over-midnight closing times
            if close_dec < open_dec:
                close_dec += 24
            row[f"{day}_open"] = open_dec
            row[f"{day}_close"] = close_dec
        except (ValueError, AttributeError):
            continue
    return row

In [30]:
hours_df = pd.DataFrame(phl_restaurants['hours'].apply(parse_hours).tolist())
hours_df = pd.concat(
    [phl_restaurants[['business_id', 'name']], hours_df], axis=1
)
hours_df.shape

(2799, 16)

## Step 3: Add derived convenience columns

In [31]:
has_any_hours = hours_df[HOURS_COLS].notna().any(axis=1)

# Number of days open per week
hours_df['days_open_per_week'] = (
    hours_df[[f"{day}_open" for day in DAYS_ORDER]].notna().sum(axis=1)
    .where(has_any_hours)
    .astype('Int64')
)

# Flag: open on weekends (Sat or Sun)
hours_df['open_weekends'] = (
    hours_df['sat_open'].notna() | hours_df['sun_open'].notna()
).astype('Int64').where(has_any_hours).astype('Int64')

# Flag: open late any night (closes after midnight, i.e. close > 24)
hours_df['open_late'] = (
    hours_df[[f"{day}_close" for day in DAYS_ORDER]].gt(24).any(axis=1)
).astype('Int64').where(has_any_hours).astype('Int64')

In [32]:
print(hours_df.shape)
hours_df.info()

(2799, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2799 entries, 0 to 2798
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   business_id         2799 non-null   object 
 1   name                2799 non-null   object 
 2   mon_open            2125 non-null   float64
 3   mon_close           2125 non-null   float64
 4   tue_open            2247 non-null   float64
 5   tue_close           2247 non-null   float64
 6   wed_open            2419 non-null   float64
 7   wed_close           2419 non-null   float64
 8   thu_open            2479 non-null   float64
 9   thu_close           2479 non-null   float64
 10  fri_open            2492 non-null   float64
 11  fri_close           2492 non-null   float64
 12  sat_open            2414 non-null   float64
 13  sat_close           2414 non-null   float64
 14  sun_open            2138 non-null   float64
 15  sun_close           2138 non-null   float64


In [33]:
hours_df.head()

,business_id,name,mon_open,mon_close,tue_open,tue_close,wed_open,wed_close,thu_open,thu_close,fri_open,fri_close,sat_open,sat_close,sun_open,sun_close,days_open_per_week,open_weekends,open_late
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,7.0,20.0,7.0,20.0,7.0,20.0,7.0,20.0,7.0,21.0,7.0,21.0,7.0,21.0,7,1,0
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,NaN,NaN,13.5,22.0,13.5,22.0,13.5,22.0,13.5,23.0,13.5,23.0,13.5,22.0,6,1,0
2,ROeacJQwBeh05Rqg7F6TCg,BAP,11.5,20.5,11.5,20.5,11.5,20.5,11.5,20.5,11.5,20.5,11.5,20.5,NaN,NaN,6,1,0
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,0.0,0.0,NaN,NaN,16.0,22.0,16.0,22.0,16.0,19.0,11.0,23.0,11.0,20.0,6,1,0
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,1,0


## Step 4: Coverage check

In [34]:
coverage = pd.Series({
    day: hours_df[f"{day}_open"].notna().mean() * 100
    for day in DAYS_ORDER
}).round(1)
print("% of restaurants with hours data per day:")
print(coverage.to_string())

print(f"\nRows with no hours data: {(~has_any_hours).sum()}")

% of restaurants with hours data per day:
mon    75.9
tue    80.3
wed    86.4
thu    88.6
fri    89.0
sat    86.2
sun    76.4

Rows with no hours data: 291


## Step 5: Merge with existing Restaurants Table

In [35]:
phl_restaurants = phl_restaurants.merge(
    hours_df, on=['business_id', 'name'], how='left'
)
phl_restaurants.shape

(2799, 77)



---



# Clean Up
The columns `attributes`, `categories`, and `hours` have been parsed and can be removed. The original columns with location data can be removed and will remain in `phl_restaurants_raw.csv` as metadata.

In [36]:
phl_restaurants.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,neighborhood,rating,...,thu_close,fri_open,fri_close,sat_open,sat_close,sun_open,sun_close,days_open_per_week,open_weekends,open_late
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,Chinatown,4.0,...,20.0,7.0,21.0,7.0,21.0,7.0,21.0,7,1,0
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,Old City,4.0,...,22.0,13.5,23.0,13.5,23.0,13.5,22.0,6,1,0
2,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,Hawthorne,4.5,...,20.5,11.5,20.5,11.5,20.5,NaN,NaN,6,1,0
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,901 N Delaware Ave,Philadelphia,PA,19123,39.962582,-75.135657,Northern Liberties,3.5,...,22.0,16.0,19.0,11.0,23.0,11.0,20.0,6,1,0
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,3604 Chestnut St,Philadelphia,PA,19104,39.954573,-75.194894,University City,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,1,0


In [39]:
phl_restaurants.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'neighborhood', 'rating', 'review_count',
       'popularity', 'ranking_score', 'attributes', 'categories', 'hours',
       'delivery', 'outdoor_seating', 'accepts_credit_cards', 'parking_garage',
       'parking_street', 'parking_validated', 'parking_lot', 'parking_valet',
       'bike_parking', 'price_range', 'take_out', 'has_wifi', 'alcohol',
       'caters', 'reservations', 'good_for_groups', 'attire', 'has_tv',
       'ambience_touristy', 'ambience_hipster', 'ambience_romantic',
       'ambience_divey', 'ambience_intimate', 'ambience_trendy',
       'ambience_upscale', 'ambience_classy', 'ambience_casual',
       'good_for_kids', 'meal_dessert', 'meal_latenight', 'meal_lunch',
       'meal_dinner', 'meal_brunch', 'meal_breakfast', 'noise_level',
       'dogs_allowed', 'happy_hour', 'wheelchair_accessible', 'table_service',
       'dietary_restrictions', 'cuisine_group', 'cuisin

In [40]:
# Remove initial columns that have been processed:
# address, city, state, postal_code, latitude, longitude, attributes, categories, hours
cols_to_drop = ['address', 'city', 'state', 'postal_code', 'latitude', 'longitude',
                'attributes', 'categories', 'hours']
phl_restaurants.drop(columns=cols_to_drop, inplace=True)
phl_restaurants.shape

(2799, 68)

In [41]:
phl_restaurants

,business_id,name,neighborhood,rating,review_count,popularity,ranking_score,delivery,outdoor_seating,accepts_credit_cards,...,thu_close,fri_open,fri_close,sat_open,sat_close,sun_open,sun_close,days_open_per_week,open_weekends,open_late
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Chinatown,4.0,80,3,0.6361,0,0,0,...,20.0,7.0,21.0,7.0,21.0,7.0,21.0,7,1,0
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,Old City,4.0,245,5,0.6963,1,1,1,...,22.0,13.5,23.0,13.5,23.0,13.5,22.0,6,1,0
2,ROeacJQwBeh05Rqg7F6TCg,BAP,Hawthorne,4.5,205,4,0.7826,<NA>,<NA>,1,...,20.5,11.5,20.5,11.5,20.5,NaN,NaN,6,1,0
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,Northern Liberties,3.5,65,3,0.5616,1,1,1,...,22.0,16.0,19.0,11.0,23.0,11.0,20.0,6,1,0
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,University City,3.0,56,3,0.5000,1,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2794,TKPAyOWcexkpVHPCdYTNmQ,Spuntino Wood Fired Pizza,Northern Liberties,4.5,209,4,0.7840,1,1,1,...,21.0,12.0,21.0,12.0,21.0,NaN,NaN,5,1,0
2795,auwFZzfhe2pvFw43OfsAfw,Stina Pizzeria,Newbold,4.5,112,4,0.7346,1,1,1,...,22.0,16.0,22.0,16.0,22.0,12.0,22.0,6,1,0
2796,K1SsvIPfFcHniNSPc3IG7g,Flip-N-Pizza,West Poplar,4.0,16,1,0.5482,1,0,1,...,25.5,15.0,25.5,15.0,25.5,11.0,23.5,7,1,1
2797,wVxXRFf10zTTAs11nr4xeA,PrimoHoagies,Roxborough,3.0,55,3,0.5000,1,0,1,...,21.0,10.0,18.0,10.0,16.0,10.0,19.0,7,1,0


# Export Structured Dataset
The structured fields will be stored in `phl_restaurants_structured.csv`.

In [42]:
# Export to CSV
phl_restaurants.to_csv(base_path + 'phl_restaurants_structured.csv', index=False)